In [1]:
# DecodeLabs AI Internship — Project 3: AI Recommendation Logic (Tech Stack Recommender)
# Content-based filtering: map a user's skills to job roles via TF-IDF + cosine similarity.
# No raw_skills.csv was provided alongside the brief, so we build an equivalent dataset
# here, matching the example roles shown in the spec (Data Scientist, DevOps Engineer,
# Backend Developer) plus a few more so a meaningful Top-3 can emerge.

import pandas as pd

job_roles = {
    "Data Scientist": "Python SQL Machine Learning Data Analysis Statistics Pandas",
    "DevOps Engineer": "AWS Docker Kubernetes CI/CD Automation Linux Terraform",
    "Backend Developer": "Java Python SQL APIs REST Microservices Spring Boot",
    "Frontend Developer": "JavaScript React HTML CSS UI Design Web Design TypeScript",
    "Cloud Architect": "AWS Azure Cloud Computing Automation Networking Security Terraform",
    "Data Analyst": "SQL Excel Data Analysis Visualization Python Statistics Tableau",
    "Machine Learning Engineer": "Python Machine Learning TensorFlow PyTorch Data Structures Algorithms",
    "Site Reliability Engineer": "Linux Kubernetes Automation Monitoring CI/CD Cloud Computing Python",
    "Mobile App Developer": "Java Kotlin Swift Mobile Development UI Design APIs",
    "Cybersecurity Analyst": "Networking Security Linux Python Risk Assessment Automation",
}

df = pd.DataFrame(list(job_roles.items()), columns=["job_role", "skills"])
df.to_csv("/kaggle/working/raw_skills.csv", index=False)

print(f"Loaded {len(df)} job roles as recommendation items.")
df

Loaded 10 job roles as recommendation items.


,job_role,skills
0,Data Scientist,Python SQL Machine Learning Data Analysis Stat...
1,DevOps Engineer,AWS Docker Kubernetes CI/CD Automation Linux T...
2,Backend Developer,Java Python SQL APIs REST Microservices Spring...
3,Frontend Developer,JavaScript React HTML CSS UI Design Web Design...
4,Cloud Architect,AWS Azure Cloud Computing Automation Networkin...
5,Data Analyst,SQL Excel Data Analysis Visualization Python S...
6,Machine Learning Engineer,Python Machine Learning TensorFlow PyTorch Dat...
7,Site Reliability Engineer,Linux Kubernetes Automation Monitoring CI/CD C...
8,Mobile App Developer,Java Kotlin Swift Mobile Development UI Design...
9,Cybersecurity Analyst,Networking Security Linux Python Risk Assessme...


In [2]:
# Step 1: Ingestion — capture real user input. Spec requires a minimum of 3 skills.
# Run this cell live and type your answers; this won't survive a headless
# "Save & Run All" commit, same as Project 1's chat loop.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Tech Stack Recommender — enter at least 3 skills or interests of yours.")
print("Press Enter on a blank line once you've entered 3 or more.\n")

user_skills = []
while True:
    skill = input(f"Skill #{len(user_skills) + 1} (blank to finish): ").strip()
    if skill == "":
        if len(user_skills) >= 3:
            break
        print(f"Need at least 3 — you've entered {len(user_skills)} so far.")
        continue
    user_skills.append(skill)

user_profile_text = " ".join(user_skills)
print(f"\nProfile captured: {user_skills}")

# Step 2: Scoring — job descriptions and the user profile must share the exact
# same vocabulary space, so we fit one TfidfVectorizer on both together rather
# than fitting on jobs alone and transforming the user separately.
corpus = df["skills"].tolist() + [user_profile_text]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

job_vectors = tfidf_matrix[:-1]   # every row except the last = job roles
user_vector = tfidf_matrix[-1]    # last row = the user's profile

# Cosine similarity over Euclidean distance, per spec: it measures orientation/
# alignment rather than raw magnitude, so a short user profile isn't penalized
# just for being shorter than a job's skill list.
df["match_score"] = cosine_similarity(user_vector, job_vectors).flatten()

Tech Stack Recommender — enter at least 3 skills or interests of yours.
Press Enter on a blank line once you've entered 3 or more.



Skill #1 (blank to finish):  python, html, css
Skill #2 (blank to finish):  


Need at least 3 — you've entered 1 so far.


Skill #2 (blank to finish):  java
Skill #3 (blank to finish):  deep learning
Skill #4 (blank to finish):  



Profile captured: ['python, html, css', 'java', 'deep learning']


In [3]:
# Step 3: Sorting — push the most relevant roles to the top.
ranked = df.sort_values("match_score", ascending=False).reset_index(drop=True)

# Step 4: Filtering — truncate to Top-3 to avoid choice overload (per the deck's
# explicit goal of curing "choice overload").
TOP_N = 3
recommendations = ranked.head(TOP_N)

print(f"\nTop {TOP_N} recommended career paths for you:\n")
for i, row in recommendations.iterrows():
    print(f"{i + 1}. {row['job_role']}  (match: {row['match_score']:.1%})")

# Save the run as a graded artifact
output_path = "/kaggle/working/recommendation_results.txt"
with open(output_path, "w") as f:
    f.write("DecodeLabs Project 3 — Tech Stack Recommender Results\n")
    f.write("=" * 50 + "\n")
    f.write(f"Input skills: {user_skills}\n\n")
    f.write(f"Top {TOP_N} Recommendations:\n")
    for i, row in recommendations.iterrows():
        f.write(f"{i + 1}. {row['job_role']} (match: {row['match_score']:.1%})\n")
    f.write("\nFull ranked list (all job roles, descending score):\n")
    f.write(ranked[["job_role", "match_score"]].to_string(index=False))

print(f"\nResults saved to {output_path}")


Top 3 recommended career paths for you:

1. Frontend Developer  (match: 24.7%)
2. Data Scientist  (match: 18.4%)
3. Backend Developer  (match: 16.9%)

Results saved to /kaggle/working/recommendation_results.txt
